In [1]:
## Part 1: Configuration Audit
from pyspark.sql import SparkSession

# Create SparkSession with default configurations
spark = SparkSession.builder \
    .appName("ConfigAudit") \
    .master("local[*]") \
    .getOrCreate()

# Collect configuration values
config_values = {
    "spark.app.name": spark.conf.get("spark.app.name"),
    "spark.master": spark.conf.get("spark.master"),
    "spark.driver.memory": spark.conf.get("spark.driver.memory", "1g"),
    "spark.executor.memory": spark.conf.get("spark.executor.memory", "1g"),
    "spark.executor.cores": spark.conf.get("spark.executor.cores", "1"),
    "spark.sql.shuffle.partitions": spark.conf.get("spark.sql.shuffle.partitions", "200"),
    "spark.default.parallelism": spark.conf.get("spark.default.parallelism", "8"),
    "spark.sql.adaptive.enabled": spark.conf.get("spark.sql.adaptive.enabled", "true"),
    "spark.sql.adaptive.coalescePartitions.enabled": spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled", "true")
}

# Print configuration values
for key, value in config_values.items():
    print(f"{key}: {value}")

spark.app.name: ConfigAudit
spark.master: local[*]
spark.driver.memory: 1g
spark.executor.memory: 1g
spark.executor.cores: 1
spark.sql.shuffle.partitions: 200
spark.default.parallelism: 8
spark.sql.adaptive.enabled: true
spark.sql.adaptive.coalescePartitions.enabled: true


In [ ]:
# Expected Output table

| Setting                                      | Default Value | What It Controls                                                                 |
|----------------------------------------------|---------------|----------------------------------------------------------------------------------|
| spark.app.name                               | ConfigAudit   | Name of the application for identification in cluster manager                    |
| spark.master                                 | local[*]      | URL of the cluster manager or execution mode                                     |
| spark.driver.memory                          | 1g            | Memory allocated to the driver process (JVM heap)                                |
| spark.executor.memory                         | 1g            | Memory allocated to each executor process                                        |
| spark.executor.cores                          | 1             | Number of CPU cores per executor for parallel task execution                     |
| spark.sql.shuffle.partitions                  | 200           | Number of partitions for shuffles in SQL operations (joins, aggregations)       |
| spark.default.parallelism                      | 8             | Default number of partitions for RDD operations like reduceByKey                 |
| spark.sql.adaptive.enabled                     | true          | Enables adaptive query execution for dynamic optimization                        |
| spark.sql.adaptive.coalescePartitions.enabled  | true          | Dynamically coalesces shuffle partitions based on map output statistics          |

In [2]:
## Part 2: Master URL Comparison
import time
from pyspark.sql import SparkSession
import pandas as pd

master_urls = ["local", "local[2]", "local[4]", "local[*]"]
results = []

for master in master_urls:
    # Create session with specific master
    spark = SparkSession.builder \
        .appName(f"MasterTest-{master}") \
        .master(master) \
        .getOrCreate()

    # Determine parallelism (estimate for local[*])
    if master == "local[*]":
        parallelism = spark.sparkContext.defaultParallelism
    else:
        # Extract number from master string
        if master == "local":
            parallelism = 1
        else:
            parallelism = int(master.split('[')[1].rstrip(']'))

    # Create test data
    data = [(i % 100, i) for i in range(100000)]
    df = spark.createDataFrame(data, ["key", "value"])

    # Time the groupBy operation
    start_time = time.time()
    result = df.groupBy("key").count().collect()
    elapsed_time = time.time() - start_time

    results.append({
        "Master URL": master,
        "Parallelism": parallelism,
        "GroupBy Time (s)": round(elapsed_time, 3)
    })

    spark.stop()

# Display results
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

Master URL  Parallelism  GroupBy Time (s)
     local            1             6.759
  local[2]            2             3.550
  local[4]            4             2.678
  local[*]            2             3.164


In [ ]:
## Why might local[*] not always be the best choice for development?

Resource contention with other applications

Overhead of managing many threads

May hide performance issues that appear in distributed mode

Can cause memory pressure if too many tasks run simultaneously

Debugging becomes more complex with parallel execution

In [3]:
## Part 3: Memory Configuration Impact
from pyspark.sql import SparkSession
import time

# Test 1: 512m driver memory
print("="*50)
print("Testing with 512m driver memory")
print("="*50)

spark_small = SparkSession.builder \
    .appName("MemoryTest-Small") \
    .master("local[*]") \
    .config("spark.driver.memory", "512m") \
    .getOrCreate()

# Create progressively larger datasets
for size in [100000, 500000, 1000000]:
    print(f"\nCreating DataFrame with {size} rows...")
    data = [(i, f"value_{i}", i * 1.0) for i in range(size)]
    df = spark_small.createDataFrame(data, ["id", "name", "value"])

    try:
        start = time.time()
        # Force collection to driver
        collected = df.collect()
        elapsed = time.time() - start
        print(f"✓ Successfully collected {len(collected)} rows in {elapsed:.2f}s")
    except Exception as e:
        print(f"✗ Failed at {size} rows: {type(e).__name__} - {str(e)}")

spark_small.stop()

# Test 2: 2g driver memory
print("\n" + "="*50)
print("Testing with 2g driver memory")
print("="*50)

spark_large = SparkSession.builder \
    .appName("MemoryTest-Large") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

for size in [100000, 500000, 1000000, 5000000]:
    print(f"\nCreating DataFrame with {size} rows...")
    data = [(i, f"value_{i}", i * 1.0) for i in range(size)]
    df = spark_large.createDataFrame(data, ["id", "name", "value"])

    try:
        start = time.time()
        collected = df.collect()
        elapsed = time.time() - start
        print(f"✓ Successfully collected {len(collected)} rows in {elapsed:.2f}s")
    except Exception as e:
        print(f"✗ Failed at {size} rows: {type(e).__name__} - {str(e)}")

spark_large.stop()

Testing with 512m driver memory

Creating DataFrame with 100000 rows...
✓ Successfully collected 100000 rows in 3.58s

Creating DataFrame with 500000 rows...
✓ Successfully collected 500000 rows in 6.00s

Creating DataFrame with 1000000 rows...
✓ Successfully collected 1000000 rows in 11.01s

Testing with 2g driver memory

Creating DataFrame with 100000 rows...
✓ Successfully collected 100000 rows in 3.17s

Creating DataFrame with 500000 rows...
✓ Successfully collected 500000 rows in 4.75s

Creating DataFrame with 1000000 rows...
✓ Successfully collected 1000000 rows in 9.70s

Creating DataFrame with 5000000 rows...
✓ Successfully collected 5000000 rows in 55.47s


In [ ]:
## What happens when collect() exceeds driver memory?
# The driver throws OutOfMemoryError or crashes entirely. In local mode, this may crash the entire JVM. Symptoms include:

# java.lang.OutOfMemoryError: Java heap space

# Driver process termination

# SparkContext stops responding

#  When is collect() appropriate vs dangerous?

# Appropriate: Small result sets (<100k rows), debugging, final results for visualization

# Dangerous: Large datasets, production pipelines, when data size is unknown

# How would you choose driver memory for production?

# Estimate result set size: Calculate worst-case output size

# Add overhead: 30-50% for driver operations and metadata

# Consider concurrency: Multiple users/operations

# Monitor usage: Use Spark UI to track driver memory

# Start conservative: 8-16GB for most production workloads

# Plan for spikes: Add 25% buffer for unexpected data growth

In [ ]:
## Part 4: SparkSession Factory
from pyspark.sql import SparkSession
import logging

def create_spark_session(
    app_name: str,
    environment: str = "development",
    extra_config: dict = None
) -> SparkSession:
    """
    Create a standardized SparkSession based on environment.

    Args:
        app_name: Name of the Spark application
        environment: 'development', 'testing', or 'production'
        extra_config: Additional Spark configuration as key-value pairs

    Returns:
        Configured SparkSession
    """

    # Define environment-specific configurations
    configs = {
        "development": {
            "master": "local[*]",
            "spark.driver.memory": "2g",
            "spark.sql.shuffle.partitions": "8",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.log.level": "WARN"
        },
        "testing": {
            "master": "local[4]",
            "spark.driver.memory": "4g",
            "spark.sql.shuffle.partitions": "50",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.log.level": "WARN"
        },
        "production": {
            "master": "yarn",
            "spark.driver.memory": "8g",
            "spark.executor.memory": "16g",
            "spark.sql.shuffle.partitions": "200",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.log.level": "ERROR"
        }
    }

    if environment not in configs:
        raise ValueError(f"Environment must be one of: {list(configs.keys())}")

    # Start building the session
    builder = SparkSession.builder \
        .appName(app_name) \
        .master(configs[environment]["master"])

    # Apply all configuration settings
    for key, value in configs[environment].items():
        if key != "master":  # Master already set
            builder = builder.config(key, value)

    # Apply extra configurations (override defaults)
    if extra_config:
        for key, value in extra_config.items():
            builder = builder.config(key, value)

    # Create the session
    spark = builder.getOrCreate()

    # Set log level
    log_level = configs[environment].get("spark.log.level", "WARN")
    spark.sparkContext.setLogLevel(log_level)

    # Log configuration summary
    logger = logging.getLogger(__name__)
    logger.info(f"Created SparkSession for {environment} environment")
    logger.info(f"Master: {spark.conf.get('spark.master')}")
    logger.info(f"Driver Memory: {spark.conf.get('spark.driver.memory')}")

    return spark

# Test the factory
def test_spark_factory():
    """Test the SparkSession factory with different environments"""

    # Development environment
    dev_spark = create_spark_session("StreamPulse-Dev", environment="development")
    print("\n=== Development Environment ===")
    print(f"Master: {dev_spark.conf.get('spark.master')}")
    print(f"Driver Memory: {dev_spark.conf.get('spark.driver.memory')}")
    print(f"Shuffle Partitions: {dev_spark.conf.get('spark.sql.shuffle.partitions')}")
    dev_spark.stop()

    # Testing environment
    test_spark = create_spark_session("StreamPulse-Test", environment="testing")
    print("\n=== Testing Environment ===")
    print(f"Master: {test_spark.conf.get('spark.master')}")
    print(f"Driver Memory: {test_spark.conf.get('spark.driver.memory')}")
    print(f"Shuffle Partitions: {test_spark.conf.get('spark.sql.shuffle.partitions')}")
    test_spark.stop()

    # Production environment with extra config
    prod_spark = create_spark_session(
        "StreamPulse-Prod",
        environment="production",
        extra_config={"spark.sql.adaptive.coalescePartitions.parallelismFirst": "false"}
    )
    print("\n=== Production Environment ===")
    print(f"Master: {prod_spark.conf.get('spark.master')}")
    print(f"Driver Memory: {prod_spark.conf.get('spark.driver.memory')}")
    print(f"Executor Memory: {prod_spark.conf.get('spark.executor.memory')}")
    print(f"Extra Config: {prod_spark.conf.get('spark.sql.adaptive.coalescePartitions.parallelismFirst')}")
    prod_spark.stop()

# Run the test
if __name__ == "__main__":
    test_spark_factory()


=== Development Environment ===
Master: local[*]
Driver Memory: 2g
Shuffle Partitions: 8

=== Testing Environment ===
Master: local[4]
Driver Memory: 4g
Shuffle Partitions: 50


In [ ]:
## Part 5: Runtime Configuration Changes
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder \
    .appName("RuntimeConfigTest") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

print("=== Initial Configuration ===")
initial_shuffle = spark.conf.get("spark.sql.shuffle.partitions")
initial_driver = spark.conf.get("spark.driver.memory")
print(f"spark.sql.shuffle.partitions: {initial_shuffle}")
print(f"spark.driver.memory: {initial_driver}")

# Try changing shuffle partitions at runtime
print("\n=== Attempting to change spark.sql.shuffle.partitions ===")
try:
    spark.conf.set("spark.sql.shuffle.partitions", "4")
    new_shuffle = spark.conf.get("spark.sql.shuffle.partitions")
    print(f"✓ Successfully changed to: {new_shuffle}")
except Exception as e:
    print(f"✗ Failed: {e}")

# Try changing driver memory at runtime
print("\n=== Attempting to change spark.driver.memory ===")
try:
    spark.conf.set("spark.driver.memory", "4g")
    new_driver = spark.conf.get("spark.driver.memory")
    print(f"✓ Successfully changed to: {new_driver}")
except Exception as e:
    print(f"✗ Failed: {e}")

# Try changing other settings
test_configs = [
    "spark.sql.adaptive.enabled",
    "spark.executor.memory",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.app.name"
]

print("\n=== Testing Additional Settings ===")
for config in test_configs:
    try:
        old_value = spark.conf.get(config, "not set")
        spark.conf.set(config, "false" if old_value == "true" else "true")
        new_value = spark.conf.get(config)
        print(f"{config}: ✓ Mutable (was {old_value}, now {new_value})")
    except Exception as e:
        print(f"{config}: ✗ Immutable - {str(e)}")

spark.stop()

In [ ]:
## Runtime Mutability Classification Table
| Setting                                                | Mutable at Runtime? | Notes                                                                                |
|--------------------------------------------------------|---------------------|---------------------------------------------------------------------------------------|
| **Mutable Settings**                                   |                     |                                                                                       |
| spark.sql.shuffle.partitions                           | ✅ Yes              | Can be tuned per query/session; affects subsequent shuffles                          |
| spark.sql.adaptive.enabled                             | ✅ Yes              | AQE can be toggled during session for dynamic optimization                           |
| spark.sql.adaptive.coalescePartitions.enabled          | ✅ Yes              | Dynamic partitioning can be enabled/disabled mid-session                             |
| spark.sql.adaptive.advisoryPartitionSizeInBytes        | ✅ Yes              | Can adjust target partition size for AQE at runtime                                   |
| spark.sql.autoBroadcastJoinThreshold                   | ✅ Yes              | Can change broadcast join behavior dynamically                                       |
| spark.sql.files.maxPartitionBytes                      | ✅ Yes              | Can adjust file reading parallelism                                                  |
| spark.sql.broadcastTimeout                             | ✅ Yes              | Timeout for broadcast joins can be modified                                           |
| spark.sql.shuffle.sort.localSortMemory                 | ✅ Yes              | Memory for sorting during shuffles can be tuned                                       |
| spark.sql.adaptive.skewJoin.enabled                    | ✅ Yes              | Can enable/disable skew join optimization dynamically                                 |
| spark.sql.adaptive.localShuffleReader.enabled          | ✅ Yes              | Can toggle local shuffle reader optimization                                         |
| spark.sql.adaptive.optimizeSkewsInRebalancePartitions  | ✅ Yes              | Can enable/disable skew optimization in rebalance                                    |
| spark.sql.adaptive.coalescePartitions.parallelismFirst | ✅ Yes              | Can change partition coalescing strategy                                             |
| spark.sql.adaptive.coalescePartitions.minPartitionSize | ✅ Yes              | Minimum partition size for coalescing can be adjusted                                |
| spark.sql.adaptive.coalescePartitions.maxPartitionSize | ✅ Yes              | Maximum partition size for coalescing can be adjusted                                |
|---------------------------------------------------------------------------------------------------------------------|
| **Immutable Settings**                                 |                     |                                                                                       |
| spark.app.name                                         | ❌ No               | Set at session creation; identifies app in UI/metrics                                |
| spark.master                                           | ❌ No               | Cluster connection established at start; cannot change                               |
| spark.driver.memory                                    | ❌ No               | JVM already allocated; requires restart                                              |
| spark.driver.memoryOverhead                            | ❌ No               | Off-heap memory allocated at startup; cannot change                                   |
| spark.driver.cores                                     | ❌ No               | Driver cores fixed at JVM startup                                                    |
| spark.executor.memory                                  | ❌ No               | Executors already provisioned; requires restart                                       |
| spark.executor.memoryOverhead                          | ❌ No               | Executor off-heap memory fixed at allocation                                          |
| spark.executor.cores                                   | ❌ No               | Executor resources fixed at allocation time                                          |
| spark.executor.instances                               | ❌ No               | Number of executors fixed at allocation (unless dynamic allocation enabled)          |
| spark.executor.heartbeatInterval                       | ❌ No               | Set at executor startup; cannot be changed dynamically                                |
| spark.sql.warehouse.dir                                | ❌ No               | Warehouse location must be set at session creation                                   |
| spark.sql.catalogImplementation                        | ❌ No               | Catalog type (hive/in-memory) set at startup                                          |
| spark.sql.extensions                                   | ❌ No               | SQL extensions loaded at initialization                                               |
| spark.sql.sources.parallelPartitionDiscovery.parallelism | ❌ No             | Parallelism for partition discovery set at startup                                    |
| spark.serializer                                       | ❌ No               | Serializer (Java/Kryo) set at context creation                                        |
| spark.kryo.registrator                                 | ❌ No               | Kryo registrator classes loaded at startup                                            |
| spark.kryo.classesToRegister                           | ❌ No               | Classes for Kryo serialization registered at startup                                  |
| spark.kryoserializer.buffer.max                        | ❌ No               | Kryo buffer size fixed at initialization                                              |
| spark.shuffle.compress                                 | ❌ No               | Shuffle compression enabled/disabled at startup                                       |
| spark.io.compression.codec                             | ❌ No               | Compression codec set at initialization                                               |
| spark.authenticate                                     | ❌ No               | Security authentication enabled/disabled at startup                                   |
| spark.akka.frameSize                                   | ❌ No               | Akka frame size fixed at startup (for Spark 2.x)                                      |
| spark.rpc.message.maxSize                              | ❌ No               | RPC message size fixed at startup                                                     |
| spark.network.timeout                                  | ❌ No               | Network timeout values require restart                                                 |
| spark.executor.extraJavaOptions                        | ❌ No               | JVM options for executors set at launch                                               |
| spark.driver.extraJavaOptions                          | ❌ No               | JVM options for driver set at launch                                                  |
| spark.python.worker.memory                             | ❌ No               | Python worker memory fixed at startup                                                 |
| spark.python.profile                                   | ❌ No               | Python profiling enabled/disabled at startup                                          |
|---------------------------------------------------------------------------------------------------------------------|
| **Conditionally Mutable**                              |                     |                                                                                       |
| spark.sql.adaptive.maxShuffledHashJoinLocalMapThreshold | ⚠️ Depends         | Mutable only if AQE is enabled                                                       |
| spark.sql.adaptive.maxNumPostShufflePartitions         | ⚠️ Depends         | Can change if AQE enabled; otherwise fixed                                            |
| spark.sql.adaptive.minNumPostShufflePartitions         | ⚠️ Depends         | Can change if AQE enabled; otherwise fixed                                            |
| spark.sql.adaptive.shuffle.targetPostShuffleInputSize  | ⚠️ Depends         | Can change if AQE enabled; otherwise fixed                                            |
| spark.sql.adaptive.nonEmptyPartitionRatio              | ⚠️ Depends         | Mutable only when dynamic partition pruning is active                                 |
| spark.sql.adaptive.logLevel                             | ⚠️ Depends         | Can change logging level if log4j allows dynamic updates                              |
| spark.sql.adaptive.forceApply                           | ⚠️ Depends         | Can be toggled only in specific query contexts                                        |